In [ ]:
# Lab type: debug
# Course: AI401 — AI Applications with LLMs
# Lesson: Cost, Latency, and Throughput: Engineering the LLM Budget
# Task: Find and fix 3 bugs in a cost estimation and caching system

# Lab: Debugging a Cost and Caching Pipeline

The pipeline below estimates API costs, caches responses, and constructs prompts for prefix-cache efficiency. It runs without Python errors, but each of the three functions contains a bug that silently produces wrong behaviour in production.

**Your task:** Find the 3 bugs and write fixes. The test harness exposes each failure.

## Setup

In [ ]:
import hashlib
import json
import tempfile
from pathlib import Path

## Function 1: `estimate_pipeline_cost`

In [ ]:
PRICE_PER_MILLION_INPUT = 3.00    # USD — Claude Sonnet input
PRICE_PER_MILLION_OUTPUT = 15.00  # USD — Claude Sonnet output


def estimate_pipeline_cost(
    n_records: int,
    system_prompt_tokens: int,
    avg_input_tokens: int,
    avg_output_tokens: int,
) -> dict:
    """Estimate total API cost for a batch pipeline run."""
    total_input = n_records * (system_prompt_tokens + avg_input_tokens)
    total_output = n_records * avg_output_tokens

    input_cost = total_input * PRICE_PER_MILLION_INPUT
    output_cost = total_output * PRICE_PER_MILLION_OUTPUT

    return {
        "n_records": n_records,
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "input_cost_usd": round(input_cost, 4),
        "output_cost_usd": round(output_cost, 4),
        "total_cost_usd": round(input_cost + output_cost, 4),
    }

## Function 2: `LLMCache`

A persistent disk cache that maps (system_prompt, user_message) → response text.

In [ ]:
class LLMCache:
    def __init__(self, cache_dir: Path):
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(exist_ok=True)

    def _cache_key(self, system_prompt: str, user_message: str) -> str:
        content = f"{system_prompt}|||{user_message}"
        return str(hash(content))

    def get(self, system_prompt: str, user_message: str) -> str | None:
        key = self._cache_key(system_prompt, user_message)
        cache_file = self.cache_dir / f"{key}.json"
        if cache_file.exists():
            return json.loads(cache_file.read_text())["output"]
        return None

    def set(self, system_prompt: str, user_message: str, output: str) -> None:
        key = self._cache_key(system_prompt, user_message)
        cache_file = self.cache_dir / f"{key}.json"
        cache_file.write_text(json.dumps({"output": output}))

## Function 3: `build_extraction_prompt`

Builds a prompt for an invoice extraction pipeline. Designed to take advantage of provider-side prefix caching.

In [ ]:
def build_extraction_prompt(invoice_text: str, system_prompt: str) -> tuple[str, list]:
    """
    Return (system, messages) ready for the Anthropic API.
    Structured to benefit from prefix caching.
    """
    user_message = f"{invoice_text}\n\n{system_prompt}"
    return system_prompt, [{"role": "user", "content": user_message}]

## Test harness

Run these cells to see each failure mode.

In [ ]:
# Test 1: cost estimate should be in USD (small numbers for 10k records)
# Expected: ~$24 input cost, ~$7.50 output cost
estimate = estimate_pipeline_cost(
    n_records=10_000,
    system_prompt_tokens=600,
    avg_input_tokens=200,
    avg_output_tokens=50,
)
print('Cost estimate:', estimate)

# Sanity check: total cost should be roughly $31.50, not $31_500_000
if estimate['total_cost_usd'] > 100_000:
    print(f'FAIL: total_cost_usd={estimate["total_cost_usd"]:,.2f} is implausibly large')
    print('Hint: check the units used in the cost formula')
elif estimate['total_cost_usd'] < 1:
    print('FAIL: cost estimate is implausibly small')
else:
    print(f'PASS: estimate looks plausible (${estimate["total_cost_usd"]:.2f})')

In [ ]:
# Test 2: LLMCache must be deterministic across Python process restarts
# Python's hash() is randomised by PYTHONHASHSEED — simulate two separate processes

import os, subprocess, sys, tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    probe_script = f'''
import sys, json, hashlib
from pathlib import Path

class LLMCache:
    def __init__(self, cache_dir):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
    def _cache_key(self, system_prompt, user_message):
        content = f"{{system_prompt}}|||{{user_message}}"
        return str(hash(content))          # <- buggy line
    def set(self, sp, um, out):
        key = self._cache_key(sp, um)
        (self.cache_dir / f"{{key}}.json").write_text(json.dumps({{"output": out}}))
    def get(self, sp, um):
        key = self._cache_key(sp, um)
        f = self.cache_dir / f"{{key}}.json"
        return json.loads(f.read_text())["output"] if f.exists() else None

cache = LLMCache(sys.argv[1])
cache.set("system", "message", "cached-response")
result = cache.get("system", "message")
print("hit" if result else "miss")
'''

    # Write the probe script
    script_path = Path(tmpdir) / 'probe.py'
    script_path.write_text(probe_script)
    cache_dir = Path(tmpdir) / 'cache'

    # Run twice with different hash seeds (simulates two Python process starts)
    env1 = {**os.environ, 'PYTHONHASHSEED': '1'}
    env2 = {**os.environ, 'PYTHONHASHSEED': '2'}

    # Process 1: write to cache
    subprocess.run([sys.executable, str(script_path), str(cache_dir)],
                   env=env1, check=True, capture_output=True)

    # Process 2: try to read — will it find the entry?
    r = subprocess.run([sys.executable, str(script_path), str(cache_dir)],
                       env=env2, capture_output=True, text=True)
    print(f'Process 2 cache result: {r.stdout.strip()!r}')
    if r.stdout.strip() == 'miss':
        print('FAIL: cache miss across process restarts — hash() is not deterministic')
        print('Hint: use hashlib.sha256() instead of hash()')
    else:
        print('PASS: cache hit across process restarts')

In [ ]:
# Test 3: inspect the prompt structure returned by build_extraction_prompt
INVOICE_SYSTEM = (
    "Extract the invoice number and total amount from the following invoice. "
    "Return a JSON object with fields 'invoice_number' (string) "
    "and 'total_amount' (number). Return only the JSON object."
)
invoice = "Invoice #INV-2024-0042\nDate: 2024-11-15\nTotal due: $1,250.00"

system, messages = build_extraction_prompt(invoice, INVOICE_SYSTEM)

user_content = messages[0]['content']
print('System prompt (first 80 chars):', system[:80])
print()
print('User message (first 120 chars):', user_content[:120])
print()
# For prefix caching to work, the user message must contain ONLY the variable content
# The static system prompt must NOT appear in the user message
if INVOICE_SYSTEM[:40] in user_content:
    print('FAIL: static system prompt text appears inside the user message')
    print('Hint: prefix caching caches the system prompt separately — do not duplicate it in user turn')
else:
    print('PASS: user message contains only the variable document')

## Find the bugs

In [ ]:
# Bug 1 — in estimate_pipeline_cost
#
# Diagnosis:
#
#
# Fix:
def estimate_pipeline_cost_fixed(
    n_records: int,
    system_prompt_tokens: int,
    avg_input_tokens: int,
    avg_output_tokens: int,
) -> dict:
    pass  # implement here

In [ ]:
# Bug 2 — in LLMCache._cache_key
#
# Diagnosis:
#
#
# Fix (rewrite _cache_key to use a deterministic hash):
class LLMCacheFixed(LLMCache):
    def _cache_key(self, system_prompt: str, user_message: str) -> str:
        pass  # implement here

In [ ]:
# Bug 3 — in build_extraction_prompt
#
# Diagnosis:
#
#
# Fix (return prompt structure where only variable content is in the user message):
def build_extraction_prompt_fixed(invoice_text: str, system_prompt: str) -> tuple[str, list]:
    pass  # implement here